In [ ]:
# This line installs the latest version of cvxpy, which is needed to use the
# IPOPT solver below. After the next cvxpy release, just importing cvxpy will be
# sufficient.
!pip install git+https://github.com/cvxpy/cvxpy.git

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# Problem data.
m1 = 1 # cart mass
m2 = 1 # pole mass
g = 10 # gravity acceleration
l = 1 # rod length
h = .05 # discretization time step
K = 60 # number of time steps
u_max = 10 # maximum absolute value of control input
q1_max = 1 # maximum absolute value of cart position

# Initial and final state.
x0 = np.array([0, np.pi, 0, 0])
xK = np.array([0, 0, 0, 0])

In [ ]:
# Cart-pole dynamics in state-space form.
def f(x, u):
  s = cp.sin(x[1])
  c = cp.cos(x[1])
  den = m1 + m2 * s ** 2
  dq1 = x[2]
  dq2 = x[3]
  ddq1 = (m2 * s * (g * c - l * x[2] ** 2) + u) / den
  ddq2 = ((m1 + m2) * g * s - m2 * l * c * s * x[2] ** 2 + c * u) / l / den
  return cp.hstack([dq1, dq2, ddq1, ddq2])

In [ ]:
# Complete the optimal control problem below. Ensure that:
# - states in the array x and control inputs in the array u are coupled by the
#   explicit Euler approximation of the cart-pole dynamics with time step h
# - intial state is equal to x0 and final state is equal to xK
# - control inputs have value between -u_max and u_max
# - cart positions have value between -q1_max and q1_max

# Variables.
x = cp.Variable((K + 1, 4)) # state at each time step x[0], ..., x[K]
u = cp.Variable(K) # input at each time step u[0], ..., u[K-1]

# Initialize empty list of constraints.
constraints = []

# Cart-pole dynamics.
### YOUR CODE HERE
### YOUR CODE HERE

# Initial and final state constraints.
### YOUR CODE HERE
### YOUR CODE HERE

# Bounds on control input.
### YOUR CODE HERE
### YOUR CODE HERE

# Bounds on cart position.
### YOUR CODE HERE
### YOUR CODE HERE

# Objective function.
obj = h * cp.sum_squares(u)

# Initial guess.
u.value = np.zeros(K)
x.value = np.linspace(x0, xK, K + 1)

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(nlp=True, solver=cp.IPOPT)

In [ ]:
# Pot optimal control input.
time_steps = np.arange(K)
plt.plot(time_steps, u.value)
plt.plot(time_steps, [u_max] * K, color='red')
plt.plot(time_steps, [-u_max] * K, color='red')
plt.xlabel('Time step')
plt.ylabel(r'Control input $u$')
plt.grid()

In [ ]:
# Pot optimal cart position.
time_steps = np.arange(K + 1)
plt.plot(time_steps, x[:, 0].value)
plt.plot(time_steps, [q1_max] * (K + 1), color='red')
plt.plot(time_steps, [-q1_max] * (K + 1), color='red')
plt.xlabel('Time step')
plt.ylabel(r'Cart position $q_1$')
plt.grid()

In [ ]:
# Animate cart-pole motion.
fig, ax = plt.subplots()
ax.set_xlim(-3, 3)  # adjust animation width to your trajectory
ax.set_ylim(- l - .2, l + .2)
ax.set_aspect('equal')
ax.grid()

# Cart and pole represented as lines.
cart_width = 0.3
cart, = ax.plot([], [], lw=20)
pole, = ax.plot([], [], marker='o', lw=2)
def init():
    cart.set_data([], [])
    pole.set_data([], [])
    return cart, pole

# Animation update function.
def update(frame):
    q1 = x.value[frame, 0]
    q2 = x.value[frame, 1]
    cart_x = [q1 - cart_width / 2, q1 + cart_width / 2]
    pole_x = [q1, q1 + l * np.sin(q2)]
    pole_y = [0, l * np.cos(q2)]
    cart.set_data(cart_x, [0, 0])
    pole.set_data(pole_x, pole_y)
    return cart, pole

# Create animation.
ani = FuncAnimation(fig, update, frames=K+1, init_func=init, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())